# 02. 高级量化方法与配置

本章把 Quark LLM 量化实战前最重要的配置概念串起来。它参考 AMD Quark 的 LLM 配置指南，本教程只涉及 Quark Pytorch, 不涉及 Quark Onnx。

这一章不跑重任务，重点是理解：

- `LLMTemplate` 如何帮我们快速生成 LLM 量化配置。
- SmoothQuant、AWQ、GPTQ、Rotation 等高级量化方法分别解决什么问题。
- 如果不使用模板，从零配置 PyTorch 量化时要关心哪些维度。

## 1. LLMTemplate：LLM 量化配置的快捷入口

对于 LLM，Quark 推荐通过 `LLMTemplate` 创建量化配置。它把常见模型结构的层名、KV cache 层、默认排除层、高级算法配置等封装起来。

典型流程是：

```python
from quark.torch import LLMTemplate

template = LLMTemplate.get("llama")
config = template.get_config(
    scheme="int4_wo_128",
    algorithm="awq",
)
```

这比从底层 `QTensorConfig`、`QLayerConfig`、`QConfig` 一层层手写更适合大多数 LLM 实战。

## 2. LLMTemplate 支持的常见量化方案

Quark 的 LLM 配置页列出了一组常用 scheme。课程里先重点关注这些：

| scheme | 描述 | 常见用途 |
| --- | --- | --- |
| `int4_wo_128` / `int4_wo_64` / `int4_wo_32` | INT4 weight-only，对称，per-group | AWQ / 低显存推理 |
| `uint4_wo_128` / `uint4_wo_64` / `uint4_wo_32` | UINT4 weight-only，非对称，per-group | AWQ / 权重量化 |
| `int8` | INT8，per-tensor，静态量化 | W8A8 / SmoothQuant 对照 |
| `fp8` | FP8 E4M3，per-tensor，静态量化 | MI300/MI350/MI355 等 FP8 路线 |
| `ptpc_fp8` | weight per-channel static，activation per-token dynamic | 面向 vLLM 优化的 FP8 路线 |
| `mxfp4` | OCP MXFP4，per-group，group size 32，动态量化 | MI350/MI355 上的 MXFP4 路线 |

我们当前课程主线先走：

- `int4_wo_128 + awq`
- `int8 + smoothquant`

## 3. 高级量化算法怎么选？

这些算法不是新的 `ModelQuantizer`，而是挂在 `QConfig.algo_config` 或 `LLMTemplate.get_config(..., algorithm=...)` 上的高级处理步骤。

| 方法 | 解决的问题 | 适合场景 |
| --- | --- | --- |
| **SmoothQuant** | activation outlier 让 W8A8 难量化 | INT8 / W8A8，想量化 activation 时 |
| **AutoSmoothQuant** | 自动搜索更合适的 smoothing scale | 不想手调 SmoothQuant alpha 时 |
| **AWQ** | 低 bit 权重量化时保护重要通道 | INT4 weight-only，vLLM 低显存推理 |
| **GPTQ** | 用 Hessian/二阶信息做列级误差补偿 | 更重但通常更准的 weight-only PTQ |
| **Qronos / GPTAQ** | Hessian 族更高级变体 | 研究/高精度恢复场景 |
| **Rotation / QuaRot** | 通过旋转降低 outlier，改善低 bit 量化 | 极低 bit、KV cache、W8A8 outlier 场景 |

本课程先实战 SmoothQuant 和 AWQ，因为它们最容易形成“量化导出 → vLLM 验证”的闭环。

## 4. 一个高级 LLMTemplate 配置长什么样？

下面这个例子展示了 Quark LLM 配置页里提到的多个能力：全局 scheme、高级算法、KV cache、attention、按层覆盖和排除层。

```python
from torch import nn
from quark.torch import LLMTemplate

llama_template = LLMTemplate.get("llama")

config = llama_template.get_config(
    scheme="int4_wo_128",          # 全局量化方案
    algorithm="awq",               # 高级量化算法
    kv_cache_scheme="fp8",         # KV cache 量化，目前 LLMTemplate 中主要是 fp8
    min_kv_scale=1.0,              # KV cache scale 下限
    attention_scheme="fp8",        # attention 量化，目前主要是 fp8
    layer_config={                 # 按层名覆盖量化方案
        "*.mlp.gate_proj": "mxfp4",
        "*.mlp.up_proj": "mxfp4",
        "*.mlp.down_proj": "mxfp4",
    },
    layer_type_config={            # 按层类型覆盖量化方案
        nn.LayerNorm: "fp8",
    },
    exclude_layers=["lm_head"],    # 排除不量化的层
)
```

需要记住两点：

- layer-wise / layer-type-wise 配置可以覆盖 global scheme。
- 算法顺序很重要；如果传入算法列表，Quark 会按列表顺序执行。

## 5. 从零配置 PyTorch 量化：四层对象

如果不用 `LLMTemplate`，Quark 的 PyTorch 量化配置可以从底层对象开始搭：

1. **`QTensorConfig`**：描述某个 tensor 怎么量化，例如 `dtype`、observer、qscheme、是否 dynamic、是否 symmetric、group size。
2. **`QLayerConfig`**：描述某类 module 的输入、输出、weight、bias 分别使用什么 tensor quant config。
3. **`AlgoConfig`**：可选，高级算法配置，例如 `AWQConfig`、`SmoothQuantConfig`、`GPTQConfig`、`RotationConfig`、`QronosConfig`。
4. **`QConfig`**：模型级总配置，组合 global config、layer config、exclude、algo config 等。

直觉上：

```text
QTensorConfig  ->  一个 tensor 怎么量化
QLayerConfig   ->  一个 layer 的哪些 tensor 要量化
AlgoConfig     ->  量化前/量化中要不要做高级优化
QConfig        ->  整个模型的量化方案
```

## 6. Calibration Methods：校准方法

校准的目标是用代表性数据估计量化参数，比如 min/max、scale、zero point。

Quark 文档里提到三类方法：

- **MinMax**：记录 tensor 的运行最小值和最大值，用它们计算量化参数。简单直接，是最常见的基线。
- **Percentile**：基于静态 histogram 的分位数信息做缩放，而不是直接使用绝对 min/max。它更适合处理 outlier。
- **MSE**：通过最小化量化前后误差来寻找量化参数，更关注输出误差或重构误差。

在 LLM 里，如果 activation outlier 很明显，纯 MinMax 可能会被极端值牵着走，这也是 SmoothQuant、AWQ 这类方法有价值的原因之一。

## 7. Calibration Datasets：校准数据集

Quark 使用 PyTorch `DataLoader` 进行校准。校准数据不需要标签，但需要尽量代表真实推理输入分布。

常见输入形式包括：

- `torch.Tensor`
- `list[torch.Tensor]`
- `list[dict[str, torch.Tensor]]`
- `dict[str, torch.Tensor]`，通常需要自定义 `collate_fn`

LLM 示例里常见校准集包括：

- `pileval`
- `wikitext`
- `pileval_for_awq_benchmark`
- `wikitext_for_gptq_benchmark`

课程里第一轮会把 `num_calib_data` 设得很小，只验证控制流；正式实验再逐步增大。

## 8. Quantization Strategies：量化策略

Quark PyTorch 文档把 PTQ 策略分成三类：

- **Post Training Weight-Only Quantization**：提前量化权重，推理时 activation 仍保持原浮点类型。AWQ / INT4 weight-only 属于这条直觉。
- **Post Training Static Quantization**：权重和 activation 都量化。需要校准数据来确定 activation 的量化参数。
- **Post Training Dynamic Quantization**：权重提前量化，activation 在运行时动态量化。适合 activation 分布难以提前确定或变化较大的情况。

本课程的两条主线：

- AWQ：更接近 weight-only。
- SmoothQuant：更接近静态 W8A8 / activation 也参与量化的路线。

## 9. Quantization Schemes：量化粒度

Quark 支持：

- **per-tensor**：整个 tensor 共用一个 scale。配置简单，但对 outlier 更敏感。
- **per-channel**：每个 channel 单独一组量化参数。通常比 per-tensor 更准，但参数更多。
- **per-group**：把 tensor 切成小 group 分别量化。INT4 weight-only 常用这种方式，在精度和压缩率之间折中。

比如 `int4_wo_128` 中的 `128` 就是 group size。

## 10. Quantization Symmetry：对称与非对称

对整数类型来说，量化还要考虑对称性：

- **symmetric quantization**：zero point 通常是 0，公式更简单，硬件实现友好。
- **asymmetric quantization**：zero point 可以偏移，让量化后的 0 表示真实值中不一定为 0 的位置，能更灵活覆盖非对称分布。

课程里会见到：

- `int4_wo_*`：对称 INT4 weight-only。
- `uint4_wo_*`：非对称 UINT4 weight-only。

## 11. 本章小结

进入实战前，先记住这张经验地图：

```text
目标硬件 / 推理框架
  -> 选择 quantization strategy
  -> 选择 scheme / dtype / symmetry / granularity
  -> 选择 calibration method 和 calibration dataset
  -> 可选高级算法：SmoothQuant / AWQ / GPTQ / Rotation
  -> 用 LLMTemplate 或从 QTensorConfig 开始配置
  -> ModelQuantizer.quantize_model()
```

下一章开始，我们先做环境体检和未量化 baseline，然后进入 SmoothQuant / AWQ 实战。